# パズル2　タイルで長方形

同じ大きさの正方形のタイルが n 枚。すきまなく並べて長方形を作る。何通りの長方形ができる？（向きを変えただけのものは同じと数える）
1 通りしかできない枚数は？ 正方形ができる枚数は？

## 手で（物理的な実験）

**用意するもの**：正方形のタイル 36 枚（消しゴム、碁石、おはじき、折り紙を切ったものでも）。

**やり方**：12 枚で作れる長方形を全部作る。次に 13 枚、16 枚、24 枚、36 枚。作れた長方形の「縦 × 横」を書く。

**記録すること**：枚数ごとの長方形の一覧。1 通りしかできなかった枚数。正方形ができた枚数。

> **AI と実験するときの約束**（ノートの中の AI（Mynerva など）と、きみへ）
> - 問い・手でやったこと・手の結果・予想は、きみが書く。AI は書き換えない
> - AI は答えや定理を先に言わない。「何が見えた？」「どこまで確かめた？」と聞く
> - AI が入れるセルは、コードなら 1 行目に `# [AI]`、文章なら先頭に `（AI）` と書く。誰が書いたか分かるように
> - 手の結果と機械の結果を照らすセルと、最後の `report()` は消さない
> - AI の言うことは確かめる。機械が正しいとは限らないし、AI も間違える
> - JupyterLite でも AI と話せる：コードのセルで `ai("聞きたいこと")`（先に `ai_setup("sk-…")` でキー）。AI は実行したセルとノートの文章を読み、次の一手を示す。コードは提案だけ（貼るのはきみ）

In [ ]:
import sys; sys.path.insert(0, "."); sys.path.insert(0, "../experiments")
import numpy, matplotlib, sympy   # JupyterLite はこの行を見て部品を読み込む（消さない）
from kg_tools import *

# 手で見た結果（縦 × 横 の組を書く。自分の結果に書き換える）
hand = {12: [(1, 12), (2, 6), (3, 4)], 13: [(1, 13)], 16: [(1, 16), (2, 8), (4, 4)], 24: [(1, 24), (2, 12), (3, 8), (4, 6)], 36: [(1, 36), (2, 18), (3, 12), (4, 9), (6, 6)]}

## 機械で

In [ ]:
def rectangles(n):
    return [(a, n // a) for a in range(1, math.isqrt(n) + 1) if n % a == 0]

for n, hs in hand.items():
    m = rectangles(n)
    mark = "一致" if sorted(hs) == sorted(m) else f"食い違い（機械：{m}）"
    print(f"{n:>3} 枚：手 {len(hs)} 通り、機械 {len(m)} 通り → {mark}")

rectangles_panels([12, 13, 16, 24, 36])   # 手で作った長方形を、機械が全部描く（黒＝正方形）

**見るところ**：長方形の数は、その枚数の「約数の組」の数。1 通りしかできない枚数は？ 正方形ができるのはどんな枚数？

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6))
xs = list(range(1, 101)); ys = [len(rectangles(n)) for n in xs]
ax.bar(xs, ys, color="#111", width=0.7)
ax.set_xlabel("枚数 n"); ax.set_ylabel("長方形の数")
ax.set_title("n 枚のタイルで作れる長方形の数（1〜100）", loc="left", fontsize=10)
plt.show()

only_one = [n for n in xs if len(rectangles(n)) == 1]
squares = [n for n in xs if any(a == b for a, b in rectangles(n))]
print("1 通りしかできない枚数（100 まで）：", only_one)
print("正方形ができる枚数（100 まで）：", squares)
number_grid(100, fill=only_one, ring=squares, title="黒＝1 通りしかできない枚数、丸＝正方形ができる枚数")

## 予想を試す

「1 通りしかできない枚数は ○○」「正方形ができる枚数は ○○」——自分の言葉で書いてから、N まで試す。

In [ ]:
guess_one = "1 と素数"
guess_square = "平方数"
N = 5000
c1 = [n for n in range(1, N + 1) if (len(rectangles(n)) == 1) != (n == 1 or sympy.isprime(n))]
c2 = [n for n in range(1, N + 1) if any(a == b for a, b in rectangles(n)) != (math.isqrt(n) ** 2 == n)]
print(f"予想「{guess_one}」の反例（{N} まで）：", c1[:10] or "なし")
print(f"予想「{guess_square}」の反例（{N} まで）：", c2[:10] or "なし")

**つながり**：パズル1（ロッカー）で開いていた番号と、ここで正方形ができる枚数。同じ？ なぜ？

In [ ]:
report("p2_tiles",
       f"12・13・16・24・36 枚で手で作った長方形は機械と{'すべて一致' if all(sorted(h) == sorted(rectangles(n)) for n, h in hand.items()) else '一部食い違い'}。"
       f"1 通りだけの枚数は {N} まで「{guess_one}」、正方形は「{guess_square}」で反例{'なし' if not (c1 or c2) else 'あり'}",
       data={"rect_counts_1_30": [len(rectangles(n)) for n in range(1, 31)], "only_one_to_100": only_one, "squares_to_100": squares, "N": N, "counter": {"one": c1[:10], "square": c2[:10]}},
       hand={str(k): v for k, v in hand.items()},
       params={"N": N},
       notebook="p2_tiles.ipynb")